# Does state-of-the-art wildfire spread prediction generalize to Santa Ana events?**An out-of-distribution evaluation of `Res18UTAE_T5` (WSTS+, WACV 2026) on wildland–urban-interface fires**---## AbstractRecent work established a new state of the art for next-day wildfire spread prediction on theWildfireSpreadTS (WSTS) benchmark: a UTAE with a pretrained ResNet-18 encoder reaching**AP = 0.478** under 12-fold leave-one-year-out cross-validation([Lahrichi et al., WACV 2026](https://arxiv.org/abs/2502.12003)).WSTS is dominated by summer and autumn fires in vegetated terrain. It is not obvious that a modelfit to that distribution transfers to **January Santa Ana events with a heavy wildland–urbaninterface component** — a regime that is rare in the benchmark but responsible for adisproportionate share of structure loss and fatalities.This notebook evaluates the *published* checkpoint, unmodified, on five historical Californiaevents, split into three extreme-wind/WUI cases (Palisades, Eaton, Camp) and twoin-distribution controls (Dixie, Caldor). The controls matter: they distinguish*"the model does not generalize"* from *"our input pipeline is broken."***Contributions**1. First OOD evaluation of the public WSTS+ SOTA checkpoint on Santa Ana / WUI fires.2. A reproducible adapter from operational data sources (SRTM, GridMET, NLCD, HRRR, VIIRS/FIRMS)   to WSTS's 23-band input format.3. Directional error analysis — whether predicted spread aligns with the driving wind — which   area-overlap metrics (IoU, CSI) systematically fail to surface.**Pre-registered expectation.** Strong performance on Dixie/Caldor (2021 is a WSTS *training*year) and degraded performance on Palisades/Eaton. Recording this before running results isdeliberate: it makes the result falsifiable rather than a post-hoc narrative.

---## 0. ReproducibilityEverything needed to re-run this notebook, captured at execution time. A result you cannotreproduce is an anecdote.

In [ ]:
import hashlib, json, os, platform, subprocess, sysfrom pathlib import Pathfrom datetime import datetime, timezoneimport numpy as npSEED = 42np.random.seed(SEED)REPO = Path.cwd()while REPO != REPO.parent and not (REPO / "ignis_ml").is_dir():    REPO = REPO.parentsys.path.insert(0, str(REPO / "research" / "wsts" / "src"))sys.path.insert(0, str(REPO))DATA_ROOT = Path(os.environ.get("IGNIS_DATA_ROOT", REPO / "data"))CKPT_DIR  = DATA_ROOT / "pretrained" / "trained_model_weights" / "Res18UTAE_T5"OUT_DIR   = REPO / "research" / "wsts" / "results"OUT_DIR.mkdir(parents=True, exist_ok=True)ENV = {    "utc": datetime.now(timezone.utc).isoformat(),    "python": sys.version.split()[0],    "platform": platform.platform(),    "numpy": np.__version__,    "seed": SEED,    "repo": str(REPO),    "data_root": str(DATA_ROOT),}try:    import torch    ENV["torch"] = torch.__version__    ENV["device"] = ("cuda" if torch.cuda.is_available()                     else "mps" if torch.backends.mps.is_available() else "cpu")except ImportError:    ENV["torch"] = None; ENV["device"] = Nonetry:    ENV["git_sha"] = subprocess.check_output(        ["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()except Exception:    ENV["git_sha"] = "unknown"print(json.dumps(ENV, indent=2))

### 0.1 Checkpoint provenanceWe evaluate a checkpoint we did not train. Its SHA-256 goes in the record so any publishednumber is traceable to exact weights.

In [ ]:
def sha256(p: Path, buf: int = 1 << 20) -> str:    h = hashlib.sha256()    with p.open("rb") as fh:        for chunk in iter(lambda: fh.read(buf), b""):            h.update(chunk)    return h.hexdigest()if not CKPT_DIR.exists():    print(f"Checkpoint not found at {CKPT_DIR}\n\nDownload with:\n"          f'  huggingface-cli download saadlahrichi/WSTSPlus \\\n'          f'    --include "trained_model_weights/Res18UTAE_T5/*" \\\n'          f'    --local-dir "$IGNIS_DATA_ROOT/pretrained"')else:    ckpts = sorted(CKPT_DIR.rglob("*.ckpt")) + sorted(CKPT_DIR.rglob("*.pth")) \          + sorted(CKPT_DIR.rglob("*.pt"))    for c in ckpts:        print(f"{c.relative_to(CKPT_DIR)}  {c.stat().st_size/1e6:8.1f} MB  {sha256(c)[:16]}...")    ENV["checkpoints"] = {str(c.relative_to(CKPT_DIR)): sha256(c) for c in ckpts}

---## 1. Input specificationWSTS ships 23 GeoTIFF bands; the dataloader expands them to 40 model channels (landcover isone-hot encoded to 17 classes, and a binary active-fire mask is appended).Band order is transcribed verbatim from the reference implementation into`research/wsts/src/wsts_spec.py`. **This is the single highest-risk detail in the notebook** —a wrong permutation produces a model that runs, emits plausible heatmaps, and is meaningless.

In [ ]:
import wsts_spec, presetswsts_spec.validate()print(f"{wsts_spec.N_BASE} base bands -> {wsts_spec.N_MODEL} model channels")print(f"static: {len(wsts_spec.STATIC_MODEL_IDS)}   dynamic: {len(wsts_spec.DYNAMIC_MODEL_IDS)}")print(f"degree bands (sin applied): {[wsts_spec.BASE_FEATURES[i] for i in wsts_spec.DEGREE_BANDS]}")gap = wsts_spec.gap_report()for status in ("have", "derive", "missing"):    print(f"\n{status.upper()} ({len(gap[status])})")    for line in gap[status]:        print("  ", line)

### 1.1 EventsThree extreme-wind/WUI cases and two in-distribution controls. Coordinates and reference timesare shared with `ignis_ml/scripts/eval_historical.py` so results are comparable across tracks.

In [ ]:
import pandas as pdev = pd.DataFrame([{"key": p.key, "name": p.name, "lat": p.lat, "lon": p.lon,                    "ref_time": p.ref_time, "regime": p.regime, "wui": p.wui}                   for p in presets.PRESETS])print(f"OOD: {presets.OOD_KEYS}\ncontrols: {presets.CONTROL_KEYS}")ev

---## 2. Harness validation — reproduce the published number**Do not skip this.** Before claiming a model fails on our data, we must show our loading andevaluation code reproduces its *published* score on its *own* benchmark. Otherwise a low OODscore is unattributable: model failure and harness bug are indistinguishable.Target: **AP ≈ 0.478** on WSTS (Veg feature subset, T=5, 12-fold LOYO).Requires the WSTS HDF5 dataset ([Zenodo](https://zenodo.org/records/8006177)) and the referencedataloader (`git clone https://github.com/slahrichi/WildfireSpreadTS third_party/WildfireSpreadTS`).

In [ ]:
WSTS_HDF5 = DATA_ROOT / "WildfireSpreadTS_HDF5"THIRD_PARTY = REPO / "research" / "wsts" / "third_party" / "WildfireSpreadTS"ready = WSTS_HDF5.is_dir() and THIRD_PARTY.is_dir()if not ready:    print("Harness validation SKIPPED — missing prerequisites:")    if not WSTS_HDF5.is_dir():        print(f"  dataset: {WSTS_HDF5}  (zenodo.org/records/8006177, convert to HDF5)")    if not THIRD_PARTY.is_dir():        print(f"  code:    {THIRD_PARTY}")        print("           git clone https://github.com/slahrichi/WildfireSpreadTS \\")        print(f"             {THIRD_PARTY}")    print("\nEvery downstream number is PROVISIONAL until this cell reproduces ~0.478 AP.")else:    sys.path.insert(0, str(THIRD_PARTY))    print("Prerequisites present — load the model + 2021 fold and score it here.")    print("Assert |AP - 0.478| < 0.02 before continuing.")HARNESS_VALIDATED = False   # flip to True only when the assertion above passes

---## 3. MetricsWe report **Average Precision** as the primary metric, matching the benchmark. AP isthreshold-free, which matters because the operating threshold is a deployment choice, not aproperty of the model — and because selecting on a thresholded metric was worth a 29% swing inthe WSTS+ ablation.Alongside it:- **CSI** (critical success index) — the operational fire-weather standard.- **IoU / Dice** — area agreement.- **Hausdorff distance** — boundary error in km.The last one carries this paper. **Area metrics are blind to direction.** A prediction thatgrows the right *amount* in the wrong *direction* can score respectably on IoU while beingoperationally worthless — evacuate the wrong neighborhood. Palisades v3 failed exactly this way:it pushed east along the urban edge instead of southwest with the wind.All metrics are reported with **bootstrap 95% CIs** over events. With n=5 the intervals will bewide; stating them honestly is the point.

In [ ]:
def confusion(pred: np.ndarray, obs: np.ndarray, t: float):    p, o = pred >= t, obs >= 0.5    return (float((p & o).sum()), float((p & ~o).sum()), float((~p & o).sum()))def csi(pred, obs, t=0.5):    tp, fp, fn = confusion(pred, obs, t)    return tp / d if (d := tp + fp + fn) > 0 else 0.0def iou(pred, obs, t=0.5):    return csi(pred, obs, t)          # identical for binary masksdef dice(pred, obs, t=0.5):    tp, fp, fn = confusion(pred, obs, t)    return 2 * tp / d if (d := 2 * tp + fp + fn) > 0 else 0.0def average_precision(pred: np.ndarray, obs: np.ndarray, nbins: int = 2001) -> float:    """Step-wise AP from a label-split probability histogram (matches sklearn)."""    y = (obs >= 0.5).ravel()    b = np.clip((pred.ravel() * (nbins - 1)).astype(int), 0, nbins - 1)    pos = np.bincount(b[y], minlength=nbins).astype(float)    neg = np.bincount(b[~y], minlength=nbins).astype(float)    tp = np.cumsum(pos[::-1])[::-1]; fp = np.cumsum(neg[::-1])[::-1]    tot = pos.sum()    if tot == 0:        return float("nan")    with np.errstate(divide="ignore", invalid="ignore"):        prec = np.where(tp + fp > 0, tp / (tp + fp), 0.0)        rec = tp / tot    r, p = rec[::-1], prec[::-1]    return float((p * np.diff(np.concatenate([[0.0], r]))).sum())def bootstrap_ci(vals, n=10000, alpha=0.05, seed=SEED):    v = np.asarray([x for x in vals if np.isfinite(x)], float)    if v.size == 0:        return (np.nan, np.nan, np.nan)    rng = np.random.default_rng(seed)    means = rng.choice(v, (n, v.size), replace=True).mean(axis=1)    return (float(v.mean()), float(np.percentile(means, 100*alpha/2)),            float(np.percentile(means, 100*(1-alpha/2))))# self-test on a known-separable case_rng = np.random.default_rng(0)_y = (_rng.random(60000) < 0.05)_p = np.clip(_rng.beta(2, 8, 60000) + 0.45*_y, 0, 1)print(f"AP={average_precision(_p,_y):.4f} (prevalence={_y.mean():.4f}) "      f"CSI@0.5={csi(_p,_y):.4f}")assert average_precision(_p, _y) > _y.mean(), "AP must beat prevalence on separable data"print("metric self-test OK")

### 3.1 Directional errorThe metric the literature under-reports and this study needs.For each event we compute the **centroid displacement** from the ignition/prior-fire footprint tothe predicted spread, and compare its bearing to the mean wind vector over the forecast window.`cos_alignment = +1` means predicted growth is exactly downwind; `-1` means exactly upwind.

In [ ]:
def centroid(mask: np.ndarray) -> np.ndarray:    idx = np.argwhere(mask >= 0.5)    return idx.mean(axis=0)[::-1] if idx.size else np.array([np.nan, np.nan])  # (x, y)def directional_alignment(pred, prior, u_mean, v_mean, t=0.5):    """cos angle between predicted-growth displacement and the wind vector."""    c_prior, c_pred = centroid(prior), centroid(pred >= t)    if np.isnan(c_prior).any() or np.isnan(c_pred).any():        return np.nan    d = c_pred - c_prior    w = np.array([u_mean, -v_mean])          # +v is north; image y grows south    nd, nw = np.linalg.norm(d), np.linalg.norm(w)    return float(d @ w / (nd * nw)) if nd > 1e-6 and nw > 1e-6 else np.nan# sanity: growth due east under a due-east wind must score +1_g = np.zeros((64, 64)); _g[30:34, 40:48] = 1_p0 = np.zeros((64, 64)); _p0[30:34, 20:28] = 1print("east growth / east wind :", round(directional_alignment(_g, _p0, 10.0, 0.0), 3))print("east growth / west wind :", round(directional_alignment(_g, _p0, -10.0, 0.0), 3))

---## 4. Building WSTS inputs for the presetsThe adapter assembles a `[T=5, 23, H, W]` stack per event at 375 m.**Three conversions are easy to get wrong and silent when wrong:**1. **Angles must be degrees.** Bands 7 (wind direction), 13 (aspect), 19 (forecast wind   direction) are degrees; the dataloader applies `sin(deg2rad(·))`. IgnisAI stores wind as   `u,v` and aspect as `cos/sin` — both must be converted *back*.2. **Active fire is detection *time*** (hhmm → hh), not a binary mask. The binary channel is   derived downstream by the dataloader.3. **Landcover is MODIS IGBP 1..17**, not NLCD. The crosswalk is lossy and must be documented.Bands 0–2 (VIIRS M11/I2/I1) are not in any current IgnisAI pipeline and must be pulled fromEarthdata; band 4 (EVI2) derives from I1/I2.

In [ ]:
def uv_to_speed_dir(u, v):    """Inverse of ignis_ml.src.data.transforms.wind_to_uv."""    speed = np.hypot(u, v)    direction = (np.degrees(np.arctan2(-u, -v))) % 360.0   # meteorological, FROM    return speed, direction# round-trip check against the production convertersys.path.insert(0, str(REPO / "ignis_ml"))from src.data.transforms import wind_to_uvfor spd, dr in [(10.0, 45.0), (5.0, 270.0), (12.5, 180.0), (3.0, 0.0)]:    u, v = wind_to_uv(spd, dr)    s2, d2 = uv_to_speed_dir(u, v)    assert abs(s2 - spd) < 1e-6 and abs((d2 - dr + 180) % 360 - 180) < 1e-6, (spd, dr, s2, d2)print("wind u,v <-> speed,direction round-trip OK")def aspect_cos_sin_to_degrees(ac, asin):    return np.degrees(np.arctan2(asin, ac)) % 360.0_deg = np.array([0., 90., 180., 270.])_r = np.radians(_deg)assert np.allclose(aspect_cos_sin_to_degrees(np.cos(_r), np.sin(_r)), _deg)print("aspect cos/sin -> degrees OK")

In [ ]:
def build_event_stack(preset, n_days: int = 5, res_m: float = 375.0, size: int = 128):    """Assemble [n_days, 23, size, size] at res_m for one event.    TODO(data): implement per the SOURCE_MAP in wsts_spec.py.      have    (16) — resample S3 static rasters / HRRR runtime cache      derive   (4) — slope+aspect from SRTM; EVI2 from I1,I2; NLCD->IGBP crosswalk      missing  (3) — VIIRS M11,I2,I1 via earthaccess    Returning NotImplemented rather than zeros is deliberate: a zero-filled stack    would run and produce a confidently wrong heatmap.    """    raise NotImplementedError(        "Adapter not yet implemented — see wsts_spec.SOURCE_MAP. "        "Blocking gap: VIIRS bands M11/I2/I1 (Earthdata).")for p in presets.PRESETS:    try:        build_event_stack(p)    except NotImplementedError as e:        print(f"{p.key:<12} pending: {e}")        break

---## 5. ResultsPopulated once §2 validates and §4 is implemented. Structure is fixed in advance so theanalysis cannot be steered by the numbers.### 5.1 Headline: OOD vs control

In [ ]:
RESULTS_PATH = OUT_DIR / "sota_ood_results.json"if RESULTS_PATH.exists():    res = pd.DataFrame(json.loads(RESULTS_PATH.read_text()))else:    res = pd.DataFrame(columns=["event", "regime", "step", "ap", "csi", "iou",                                "hausdorff_km", "cos_alignment"])    print("No results yet — run §2 and §4 first.")if len(res):    summary = (res.groupby("regime")                 .agg(ap=("ap", "mean"), csi=("csi", "mean"),                      hausdorff_km=("hausdorff_km", "mean"),                      cos_alignment=("cos_alignment", "mean"), n=("ap", "size")))    display(summary)    for m in ("ap", "csi", "cos_alignment"):        ood = res[res.regime == "santa_ana"][m]        ctl = res[res.regime == "summer_fall"][m]        mo, lo, hi = bootstrap_ci(ood); mc, lc, hc = bootstrap_ci(ctl)        print(f"{m:>14}  OOD {mo:.3f} [{lo:.3f},{hi:.3f}]   "              f"control {mc:.3f} [{lc:.3f},{hc:.3f}]   gap {mc-mo:+.3f}")

### 5.2 Interpretation guideWritten before the results exist, so the conclusion is constrained by the design rather thanchosen after the fact.| Outcome | Reading || --- | --- || Control AP ≈ 0.45–0.48, OOD AP materially lower | **Generalization gap confirmed.** The controls prove the pipeline works, so the drop is attributable to regime. This is the paper. || Both low (control ≪ 0.45) | **Pipeline problem, not a model problem.** Our reconstructed inputs differ too much from native WSTS. Fix §4 before claiming anything. || Both high | SOTA transfers. Fine-tune from this checkpoint instead of training from scratch — a better outcome for the product, a less interesting paper. || AP comparable but `cos_alignment` low on OOD | **The most interesting result.** Right area, wrong direction — invisible to IoU/CSI, operationally disqualifying, and precisely the v3 Palisades failure. Would justify the wind-alignment regularizer directly. |### 5.3 LimitationsState plainly:1. **n = 5 events.** Bootstrap CIs will be wide. This is a case study, not a benchmark.2. **Reconstructed inputs.** NDVI/ERC/PDSI are fire-season composites in our S3, not per-day   fields; landcover is a lossy NLCD→IGBP crosswalk; 500 m sources upsampled to 375 m. Each   plausibly depresses measured performance relative to native WSTS data, and we cannot fully   separate that from genuine OOD degradation. The in-distribution controls partially bound it.3. **Perimeter ground truth** is daily and interpolated; sub-daily timing error is not captured.4. **Single checkpoint.** No seed variance for the pretrained model — the ±0.08–0.09 AP standard   deviations reported across WSTS folds suggest event-level variance is substantial.5. **Camp is not Santa Ana proper** — it is a northern-California downslope wind event. Grouped   by regime (extreme wind + WUI), not by meteorological label.

---## References- Lahrichi, S., Bova, J., Johnson, J., Malof, J. *Improved Wildfire Spread Prediction with Time-Series Data and the WSTS+ Benchmark.* WACV 2026. [arXiv:2502.12003](https://arxiv.org/abs/2502.12003) · [weights](https://huggingface.co/saadlahrichi/WSTSPlus)- Gerard, S., Zhao, Y., Sullivan, J. *WildfireSpreadTS: A dataset of multi-modal time series for wildfire spread prediction.* NeurIPS Datasets & Benchmarks, 2023. [code](https://github.com/SebastianGer/WildfireSpreadTS) · [data](https://zenodo.org/records/8006177)- Huot, F. et al. *Next Day Wildfire Spread.* IEEE TGRS, 2022. [arXiv:2112.02447](https://arxiv.org/abs/2112.02447)- Garnot, V.S.F., Landrieu, L. *Panoptic Segmentation of Satellite Image Time Series (UTAE).* ICCV 2021.- Funk, J.V. *Boundary-Aware Uncertainty Quantification for Wildfire Spread Prediction.* 2026. [arXiv:2605.03148](https://arxiv.org/abs/2605.03148)